In [1]:
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

seed = 42
trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

## Load

In [2]:
ds = load_dataset("virattt/financial-qa-10K")

tokenizer = AutoTokenizer.from_pretrained(trained_weights)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [3]:
df = ds["train"].to_pandas()
df.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [4]:
df.apply(lambda x: x.eq("")).any()

question     True
answer       True
context      True
ticker      False
filing      False
dtype: bool

In [5]:
df[df[["answer", "context", "question"]].apply(lambda x: x.eq("")).any(axis=1)]

,question,answer,context,ticker,filing
1727,How is Alphabet investing in its future growth...,,Alphabet intends to scale up investments in re...,GOOGL,2023_10K
1733,,The main components of Alphabet Inc.'s operati...,Operating expenses include costs related to R&...,GOOGL,2023_10K
2731,,,,HD,2023_10K


## Drop empty

In [6]:
indexes = df[df[["answer", "context", "question"]].apply(lambda x: x.eq("")).any(axis=1)].index.to_list()

In [7]:
df.drop(indexes, inplace=True)

In [8]:
len(df)

6997

## Dataset Sizes

In [9]:
n_70ctx   = len(df)
n_30noctx = int(n_70ctx * 0.3 / 0.7)
dataset_size = n_70ctx + n_30noctx

split_proportion = {"train": 0.8, "val": 0.1, "test": 0.1}
train_size = int(dataset_size * split_proportion["train"])
val_size   = int(dataset_size * split_proportion["val"])
test_size  = dataset_size - train_size - val_size

## Assign Prompt Types

In [10]:
# apply prompt types in the question_only first
question_only = df.sample(n=n_30noctx, random_state=seed).copy()
question_only["prompt_type"] = "question_only"

df["prompt_type"] = "context_grounded"

new_df = pd.concat([df, question_only]).reset_index(drop=True)
new_df.head()

,question,answer,context,ticker,filing,prompt_type
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K,context_grounded
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K,context_grounded
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K,context_grounded
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K,context_grounded
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K,context_grounded


In [11]:
new_df.notna().sum()

question       9995
answer         9995
context        9995
ticker         9995
filing         9995
prompt_type    9995
dtype: int64

## Shuffle & Split

In [12]:
df_shuffled = new_df.sample(frac=1, random_state=seed).reset_index(drop=True)
display(df_shuffled.head())

train_df = df_shuffled.iloc[:train_size]
val_df   = df_shuffled.iloc[train_size:train_size + val_size]
test_df  = df_shuffled.iloc[train_size + val_size:]

# Save datasets
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

,question,answer,context,ticker,filing,prompt_type
0,How did the dividend payout rate change accord...,It decreased from 67.42% to 25.86% over a cert...,"According to the financial data, the dividend ...",BAC,2023_10K,context_grounded
1,"What was the change in other income (expense),...","Other income (expense), net changed by $204.1 ...","Other income (expense), net changed by $204.1 ...",PLTR,2023_10K,context_grounded
2,In what section of the Annual Report on Form 1...,Notes to the Consolidated Financial Statements,"Note 14, which is related to legal proceedings...",EA,2023_10K,context_grounded
3,What is the role of the Nominating and Corpora...,The role of the Nominating and Corporate Gover...,The Nominating and Corporate Governance Commit...,NVDA,2023_10K,context_grounded
4,What does Note 14 of the Notes to the Consolid...,Legal proceedings,Note 14 of the Notes to the Consolidated Finan...,EA,2023_10K,context_grounded


Train: 7996 | Val: 999 | Test: 1000


## Evaluate prompt_type per split

In [13]:
display(train_df["prompt_type"].value_counts(normalize=True))
display(train_df.notna().sum())

prompt_type
context_grounded    0.702976
question_only       0.297024
Name: proportion, dtype: float64

question       7996
answer         7996
context        7996
ticker         7996
filing         7996
prompt_type    7996
dtype: int64

In [14]:
display(val_df["prompt_type"].value_counts(normalize=True))
display(val_df.notna().sum())

prompt_type
context_grounded    0.673674
question_only       0.326326
Name: proportion, dtype: float64

question       999
answer         999
context        999
ticker         999
filing         999
prompt_type    999
dtype: int64

In [15]:
display(test_df["prompt_type"].value_counts(normalize=True))
display(test_df.notna().sum())

prompt_type
context_grounded    0.703
question_only       0.297
Name: proportion, dtype: float64

question       1000
answer         1000
context        1000
ticker         1000
filing         1000
prompt_type    1000
dtype: int64

## Save

In [16]:
from pathlib import Path

# for .py in scripts
# PROJECT_ROOT  = Path(__file__).resolve().parent.parent

# for notebooks folder
PROJECT_ROOT = Path.cwd().parent
project_root = str(PROJECT_ROOT)

project_root

'/var/home/anne/Documents/_Ironhack/alpha-crunch'

In [17]:
output_dir = PROJECT_ROOT / "data" / "fiqa"

if not output_dir.is_dir():
    output_dir.mkdir()
    

In [18]:
train_csv = "train_df.csv"
val_csv = "val_df.csv"
test_csv = "test_df.csv"

train_df.to_csv(str(output_dir / train_csv), index=False)
val_df.to_csv(str(output_dir / val_csv), index=False)
test_df.to_csv(str(output_dir / test_csv), index=False)

## Loading from data folder

In [19]:
train_df_loaded = pd.read_csv(str(output_dir / train_csv))
train_df_loaded.head()

,question,answer,context,ticker,filing,prompt_type
0,How did the dividend payout rate change accord...,It decreased from 67.42% to 25.86% over a cert...,"According to the financial data, the dividend ...",BAC,2023_10K,context_grounded
1,"What was the change in other income (expense),...","Other income (expense), net changed by $204.1 ...","Other income (expense), net changed by $204.1 ...",PLTR,2023_10K,context_grounded
2,In what section of the Annual Report on Form 1...,Notes to the Consolidated Financial Statements,"Note 14, which is related to legal proceedings...",EA,2023_10K,context_grounded
3,What is the role of the Nominating and Corpora...,The role of the Nominating and Corporate Gover...,The Nominating and Corporate Governance Commit...,NVDA,2023_10K,context_grounded
4,What does Note 14 of the Notes to the Consolid...,Legal proceedings,Note 14 of the Notes to the Consolidated Finan...,EA,2023_10K,context_grounded


In [20]:
display(train_df.notnull().sum())
train_df_loaded[train_df_loaded.isna().any(axis=1)]
train_df[train_df_loaded.isna().any(axis=1)]

question       7996
answer         7996
context        7996
ticker         7996
filing         7996
prompt_type    7996
dtype: int64

,question,answer,context,ticker,filing,prompt_type


In [21]:
val_df_loaded = pd.read_csv("../data/fiqa/val_df.csv")
val_df_loaded[val_df_loaded.isna().any(axis=1)]

,question,answer,context,ticker,filing,prompt_type


In [22]:
test_df_loaded = pd.read_csv("../data/fiqa/test_df.csv")
test_df_loaded[test_df_loaded.isna().any(axis=1)]

,question,answer,context,ticker,filing,prompt_type


## Preprocessing for training

In [ ]:
def format_row(row, tokenizer, add_answer=True):

    format = "Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}"

    try:
    
        if row["prompt_type"] == "context_grounded":
            content = format.format(context=row["context"], question=row["question"])
        elif row["prompt_type"] == "question_only":
            content = row["question"]
        else:
            raise ValueError(f"Invalid prompt_type: {row['prompt_type']}")


        chat = [{"role": "user", "content": content}]

        if add_answer: 
            chat.append({"role": "assistant", "content": row["answer"]})

            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
    
        else:
            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

    except Exception as e:
        print(f"[format_row error] {e}")
        return None


In [27]:
def build_hf_dataset(df, tokenizer, add_answer = True):

    formatted = df.apply(lambda x: format_row(x, tokenizer, add_answer=add_answer), axis=1).tolist()
    formatted = [x for x in formatted if x is not None]  # filter failed rows

    return Dataset.from_dict({"chat": formatted})

train_dataset = build_hf_dataset(train_df, tokenizer)
val_dataset   = build_hf_dataset(val_df,   tokenizer)
# test_dataset  = build_hf_dataset(test_df,  tokenizer) # if I want to validate with the same pipeline of the training

dataset_splits = {
    "train": train_dataset,
    "val":   val_dataset,
    # "test":  test_dataset,
}


In [28]:
dataset_splits

{'train': Dataset({
     features: ['chat'],
     num_rows: 7996
 }),
 'val': Dataset({
     features: ['chat'],
     num_rows: 999
 })}

## Preprocessing for generation

In [30]:
test_df_loaded = pd.read_csv(str(output_dir / test_csv))
test_df_loaded.head()

,question,answer,context,ticker,filing,prompt_type
0,What is the weighted-average useful life of th...,12.0 years,The weighted-average life of the total acquire...,INTU,2023_10K,context_grounded
1,How is RYBREVANT (amivantamab) used in the tre...,RYBREVANT is used in combination with chemothe...,RYBREVANT (amivantamab) is combined with chemo...,JNJ,2023_10K,question_only
2,What section of a document does Item 8 refer to?,Financial Statements and Supplementary Data,Item 8 refers to the section designated for Fi...,GIS,2023_10K,question_only
3,When is Chevron's concession for operating Sau...,2046,Chevron holds a concession to operate the King...,CVX,2023_10K,question_only
4,What was the strategy of the company regarding...,The company's strategy in fiscal year 2023 was...,Our strategy is to combine best-of-breed techn...,AVGO,2023_10K,question_only


In [31]:
inf_inputs = test_df_loaded.apply(lambda x: format_row(x, tokenizer, add_answer=False), axis=1).tolist()
inf_inputs = [x for x in inf_inputs if x is not None]  # filter failed rows


In [32]:
inf_inputs[0]

'<s> [INST] Use the following context to answer the question.\n\nContext: The weighted-average life of the total acquired identifiable intangible assets is 12.0 years.\n\nQuestion: What is the weighted-average useful life of the total acquired identifiable intangible assets? [/INST]'